# Example 11.4.2 / 1: A Fixed-Charge Nonlinear Allocation

Companion notebook for **11.4.2 Mixed-Integer Nonlinear Models** in the AMPL lab Documentation, section "A Fixed-Charge Nonlinear Allocation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A Fixed-Charge Nonlinear Allocation

This small model selects plants and allocates production. Its separable quadratic operating cost is convex, while binary setup decisions make the overall problem discrete.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PLANTS;                      # candidate plants
param demand > 0;                # total output required
param capacity {PLANTS} > 0;     # finite capacity per plant
param fixed_cost {PLANTS} >= 0;  # cost of opening a plant
param quad_cost {PLANTS} > 0;    # convex operating cost coefficient

var Open {PLANTS} binary;   # 1 if the plant is opened
var Produce {PLANTS} >= 0;  # output per plant

minimize TotalCost:  # fixed charges plus convex quadratic operating cost
    sum {p in PLANTS}
        (fixed_cost[p] * Open[p] + quad_cost[p] * Produce[p]^2);

subject to MeetDemand:  # total output covers demand
    sum {p in PLANTS} Produce[p] >= demand;

subject to CapacityLink {p in PLANTS}:  # produce only at an open plant
    Produce[p] <= capacity[p] * Open[p];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PLANTS"] = ["north", "south"]
ampl.param["demand"] = 7
ampl.param["capacity"] = {"north": 6, "south": 8}
ampl.param["fixed_cost"] = {"north": 5, "south": 8}
ampl.param["quad_cost"] = {"north": 1, "south": 0.5}

In [5]:
# run.py
ampl.solve(solver="bonmin", verbose=False)
print(ampl.solve_result)
print(ampl.var["Open"].to_dict())
print(ampl.var["Produce"].to_dict())

solved
{'north': 1, 'south': 1}
{'north': 2.3333333100015197, 'south': 4.6666666200006235}


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PLANTS := north south;
param demand := 7;
param capacity := north 6 south 8;
param fixed_cost := north 5 south 8;
param quad_cost := north 1 south 0.5;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver bonmin;
solve > bonmin.log;
display Open, Produce, TotalCost;


bonmin: Optimal
:     Open   Produce    :=
north   1    2.33333
south   1    4.66667
;

TotalCost = 29.3333



### Another way in amplpy

Check both layers of the answer: `Open` is exactly `1` for both plants; the recomputed cost is
$5+8+1\cdot(7/3)^2+0.5\cdot(14/3)^2=29.3333$; demand is met with $7/3+14/3=7$; and both
productions stay below their capacities 6 and 8.

The valid capacities matter even though the optimum does not change. With
`option relax_integrality 1` the binaries become continuous on $[0,1]$, and Ipopt solves the
relaxation, whose optimum is a lower bound on the MINLP's cost. With the true capacities 6 and 8
the relaxation bound is 22.939814; replacing both with an arbitrary 1000 weakens it to 16.382332,
although Bonmin, and Couenne, a global MINLP solver, both still return 29.333333 for the MINLP:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PLANTS"] = ["north", "south"]
ampl.param["demand"] = 7
ampl.param["capacity"] = {"north": 6, "south": 8}
ampl.param["fixed_cost"] = {"north": 5, "south": 8}
ampl.param["quad_cost"] = {"north": 1, "south": 0.5}

In [11]:
# run.py
for cap in ({"north": 6, "south": 8}, {"north": 1000, "south": 1000}):
    ampl.param["capacity"] = cap
    ampl.option["relax_integrality"] = 1
    ampl.solve(solver="ipopt", verbose=False)
    relaxed = round(ampl.obj["TotalCost"].value(), 6)
    ampl.option["relax_integrality"] = 0
    found = []
    for solver in ("bonmin", "couenne"):
        ampl.solve(solver=solver, verbose=False)
        found.append(round(ampl.obj["TotalCost"].value(), 6))
    print(cap["north"], cap["south"], "relaxation:", relaxed, "bonmin, couenne:", found)

6 8 relaxation: 22.939814 bonmin, couenne: [29.333333, 29.333333]
1000 1000 relaxation: 16.382332 bonmin, couenne: [29.333333, 29.333333]
